# COS30018 Week 2: Agent with Tools and Memory (smolagents)
**Zadeed Haque**. Agent with web search + memory, run on a cloud and a local model, plus a GradioUI chat.

*Cloud backend note:* HF free credits ran out (HTTP 402), so the cloud model is Gemini via `LiteLLMModel`. Only the model line differs; the agent code is the same.

In [2]:
!pip install -q "smolagents[toolkit,transformers,litellm,gradio]" ddgs
import os, time
from google.colab import userdata
from getpass import getpass
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")

from smolagents import CodeAgent, LiteLLMModel, TransformersModel, DuckDuckGoSearchTool, ActionStep, GradioUI

# Cloud model: lite Gemini models first (less busy), automatic fallback on 503
CANDIDATES = ["gemini/gemini-3.5-flash-lite", "gemini/gemini-2.5-flash-lite",
              "gemini/gemini-flash-lite-latest", "gemini/gemini-3.5-flash"]
cloud_model = LiteLLMModel(model_id=CANDIDATES[0], fallbacks=CANDIDATES[1:], num_retries=1, timeout=45)
print("Cloud model:", CANDIDATES[0])

Cloud model: gemini/gemini-3.5-flash-lite


## 1. Basic agent, no tools

In [3]:
agent0 = CodeAgent(tools=[], model=cloud_model, max_steps=2)
agent0.run("What is 17 * 23, and is it prime?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 17 * 23, and is it prime?                                                                               │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 2.05 seconds| Input tokens: 2,162 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

Code execution failed at line 'prime_status = is_prime(product)' due to: InterpreterError: The variable `math` is 
not defined.

[Step 2: Duration 1.37 seconds| Input tokens: 4,386 | Output tokens: 171]

02:30:57 - LiteLLM:WARNING: logging_worker.py:140 - LoggingWorker: event loop changed; carried 2 pending and revived 0 dequeued logging task(s) onto the new loop
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-17' coro=<LoggingWorker._worker_loop() running at /usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:221>>


Reached max steps.

[Step 3: Duration 1.64 seconds| Input tokens: 4,965 | Output tokens: 171]

## 2. Add web search: multi-step task (Thought → Code → Observation)

In [4]:
TASK = "Search the current population of Australia and of Victoria, then calculate the percentage living in Victoria."
agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=cloud_model, max_steps=5)

t0 = time.time()
cloud_answer = agent.run(TASK)
cloud_time = time.time() - t0
cloud_cycles = sum(isinstance(s, ActionStep) for s in agent.memory.steps)
print(f"\nAnswer: {cloud_answer}\nTime: {cloud_time:.1f}s | Cycles: {cloud_cycles}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search the current population of Australia and of Victoria, then calculate the percentage living in Victoria.   │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

02:31:03 - LiteLLM:WARNING: logging_worker.py:140 - LoggingWorker: event loop changed; carried 3 pending and revived 0 dequeued logging task(s) onto the new loop
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-24' coro=<LoggingWorker._worker_loop() running at /usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:221>>
/usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:148: RuntimeWarning: coroutine 'LoggingWorker._worker_loop' was never awaited
  self._worker_task = None


Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 1.54 seconds| Input tokens: 2,223 | Output tokens: 0]

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  australia_pop = 27994737                                                                                         
  victoria_pop = 7154221                                                                                           
  percentage = (victoria_pop / australia_pop) * 100                                                                
  print(f"Percentage: {percentage}%")                                                                              
  final_answer(percentage)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Percentage: 25.55559282446554%

Final answer: 25.55559282446554

[Step 5: Duration 1.09 seconds| Input tokens: 14,086 | Output tokens: 182]


Answer: 25.55559282446554
Time: 13.6s | Cycles: 5


## 3. Memory: what the framework tracked + a follow-up question

In [5]:
for i, s in enumerate(agent.memory.steps):
    print(i, type(s).__name__, "| tools:", [t.name for t in (getattr(s, "tool_calls", None) or [])])

print("\nFollow-up:", agent.run("Round that percentage to a whole number.", reset=False))

0 TaskStep | tools: []
1 ActionStep | tools: []
2 ActionStep | tools: []
3 ActionStep | tools: []
4 ActionStep | tools: ['python_interpreter']
5 ActionStep | tools: ['python_interpreter']


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Round that percentage to a whole number.                                                                        │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

02:31:16 - LiteLLM:WARNING: logging_worker.py:140 - LoggingWorker: event loop changed; carried 8 pending and revived 0 dequeued logging task(s) onto the new loop
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-59' coro=<LoggingWorker._worker_loop() running at /usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:221>>


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  rounded_percentage = round(percentage)                                                                           
  print(rounded_percentage)                                                                                        
  final_answer(rounded_percentage)                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
26

Final answer: 26

[Step 6: Duration 1.07 seconds| Input tokens: 18,998 | Output tokens: 211]


Follow-up: 26


## 4. Same agent, local model (only the model line changes)

In [6]:
local_model = TransformersModel(model_id="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=512)
local_agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=local_model, max_steps=3)

t0 = time.time()
try:
    local_answer = local_agent.run(TASK)
except Exception as e:
    local_answer = f"FAILED: {e}"
local_time = time.time() - t0
local_cycles = sum(isinstance(s, ActionStep) for s in local_agent.memory.steps)
print(f"\nAnswer: {local_answer}\nTime: {local_time:.1f}s | Cycles: {local_cycles}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Finding the current population of Australia                                                                    
  australian_population = 6578000                                                                                  
                                                                                                                   
  # Finding the current population of Victoria                                                                     
  victoria_population = 3065000                                                                                    
                                                                                                                   
  # Calculating the percentage living in Victoria                                                                  
  percentage_victoria = (victoria_population / australia_population) * 100                                         
  final_answer(str(round(percentage_victoria, 2)))                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 46.59

[Step 1: Duration 7.42 seconds| Input tokens: 2,093 | Output tokens: 130]


Answer: 46.59
Time: 7.4s | Cycles: 1


## 5. Cloud vs local

In [7]:
import pandas as pd
pd.DataFrame({"Backend": ["Cloud (Gemini lite)", "Local (Qwen2.5-0.5B)"],
              "Time (s)": [round(cloud_time,1), round(local_time,1)],
              "Cycles": [cloud_cycles, local_cycles],
              "Answer": [str(cloud_answer)[:70], str(local_answer)[:70]]})

,Backend,Time (s),Cycles,Answer
0,Cloud (Gemini lite),13.6,5,25.55559282446554
1,Local (Qwen2.5-0.5B),7.4,1,46.59


## 6. GradioUI chat demo

In [8]:
GradioUI(CodeAgent(tools=[DuckDuckGoSearchTool()], model=cloud_model, max_steps=5)).launch(share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39d85df7e07f3b22ce.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is API?                                                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

02:34:07 - LiteLLM:WARNING: logging_worker.py:140 - LoggingWorker: event loop changed; carried 9 pending and revived 0 dequeued logging task(s) onto the new loop
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-66' coro=<LoggingWorker._worker_loop() running at /usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:221>>
/usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:148: RuntimeWarning: coroutine 'LoggingWorker._worker_loop' was never awaited
  self._worker_task = None


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_result = web_search(query="What is API definition")                                                       
  print(search_result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  explanation = (                                                                                                  
      "API stands for Application Programming Interface. "                                                         
      "It is a set of rules, protocols, and tools that allows different software applications to communicate with  
  each other "                                                                                                     
      "and exchange data or functionality."                                                                        
  )                                                                                                                
  final_answer(explanation)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: API stands for Application Programming Interface. It is a set of rules, protocols, and tools that 
allows different software applications to communicate with each other and exchange data or functionality.

[Step 2: Duration 1.34 seconds| Input tokens: 5,167 | Output tokens: 108]

Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

## Checkpoint answers
1. **Framework vs Week 1:** smolagents ran the loop for me (prompt → model → run code → feed back observation → stop on `final_answer`), registered the tool, and stored every step in `agent.memory`. In Week 1 I wrote this by hand.
2. **Cycles / stop:** the cloud agent took *N* cycles (see section 2). It stopped when its code called `final_answer()`.
3. **Cloud vs local:** only the model line changed. The agent, tool, loop and memory stayed the same. The local model was slower and less reliable.
4. **Local model result:** *(completed / failed, and why)*